In [ ]:
from uuid import UUID, uuid4
from datetime import datetime
from typing import TypedDict, Dict, Any, Literal, List, Optional
import json
import duckdb
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI # Use this for Gemini
from langchain_core.tools import tool
from langchain_core.runnables import RunnablePassthrough
from langgraph.graph import StateGraph, MessagesState, START, END
from osp_sos_analyser.dbconnector import DatabaseConnector
from osp_sos_analyser.context_pack import (
    compact_message_history,
    extract_identifiers,
    format_log_digest,
    prefetch_evidence_digest,
    truncate_text,
)
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
from langchain.agents import create_agent
import re


In [ ]:
db = DatabaseConnector("sos_analysis.duckdb",read_only=True)
db_con = db.connect()
db_con.execute("SELECT * FROM os_logs LIMIT 10")
print(db_con.fetchall())

## sample query to fetch logs related to a specific UUID
db_con.execute(
    "SELECT timestamp, service, level, message FROM os_logs WHERE message LIKE ? ORDER BY timestamp",
    ["%ba4bf98f-88ab-43a7-b68b-ef16ea47e0db%"]
).fetchall()

# db.fetch_all("SELECT * FROM os_logs LIMIT 10")
# print(rows)


In [ ]:
print(db_con.execute("SELECT DISTINCT service FROM os_logs ORDER BY 1").fetchall())

In [ ]:
UUID_RE = re.compile(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', re.I)

def extract_related_ids(log_text: str, exclude: str) -> list[str]:
    ids = set(UUID_RE.findall(log_text))
    ids.discard(exclude)
    return list(ids)

In [ ]:
db_con.execute("""
    SELECT service, source_file, COUNT(*) 
    FROM os_logs 
    GROUP BY service, source_file 
    ORDER BY service, source_file
""").fetchall()

In [ ]:
# Are there any raw archive files suggesting neutron-server ran on this host at all?
db_con.execute("""
    SELECT DISTINCT source_file FROM os_logs WHERE source_file LIKE '%neutron%'
""").fetchall()

In [ ]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") or os.getenv("GROK_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY")
llm = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq",
).bind(parallel_tool_calls=False)

# gemini_key = os.getenv("GOOGLE_API_KEY")
# llm = init_chat_model(
#     model="gemini-3.5-flash",
#     model_provider="google_genai",
#     google_api_key=gemini_key
#     )
llm

# ---------------------------------------------------------
# 1. Define the Pydantic Schemas (The Output Structure)
# ---------------------------------------------------------

In [ ]:
class InvestigationEntities(BaseModel):
    investigation_id: Optional[str] = Field(default_factory=lambda: str(uuid4()))
    resource_id: Optional[str] = Field(default=None, description="Any UUID at the center of the investigation — instance, volume, port, network, image, etc.")
    resource_type: Optional[str] = Field(default=None, description="e.g. 'instance', 'volume', 'port', 'network', 'image'")
    service: Optional[str] = Field(default=None)
    problem: Optional[str] = Field(default=None)
    cluster: Optional[str] = Field(default=None)

class TimeWindow(BaseModel):
    start: Optional[str] = Field(default=None)
    end: Optional[str] = Field(default=None)

class SearchTask(BaseModel):
    service: str
    objective: str
    query: str
    priority: int

class ExpandedQuery(BaseModel):
    summary: str
    intent: str
    entities: InvestigationEntities
    keywords: List[str]
    search_queries: List[SearchTask] = Field(default_factory=list, description="A list of search tasks to be performed, each with a service, objective, query, and priority.")
    investigation_targets: List[str]
    hypotheses: List[str] = Field(default_factory=list)
    time_window: Optional[TimeWindow] = Field(default=None)

class InvestigationState(TypedDict):
    investigation_id: UUID
    raw_query: str
    expanded_plan: Optional[ExpandedQuery]
    prefetch_digest: str
    gathered_evidence: List[Dict[str, Any]]
    findings: Dict[str, Any]
    final_rca: Optional[str]
    next_node: str

class Evidence(BaseModel):
    service: str
    source: str
    summary: str
    raw_logs: str


# Prompt Template

In [ ]:
prompt_expand = ChatPromptTemplate.from_messages([
    ("system", """You are a query enrichment agent for a Red Hat OpenStack investigation workflow.
Return JSON only with the fields: summary, intent, entities, keywords, search_queries, investigation_targets, hypotheses, time_window.

CRITICAL RULES:
- `keywords` MUST include any UUIDs, hostnames, or instance names verbatim from the incident report. Do NOT replace them with concept words like 'VM' or 'creation'.
- `keywords` should be lowercase tokens that would actually appear in OpenStack logs (e.g. 'failed', 'error', 'spawn', 'build', the UUID itself).
- `entities.service` MUST be one of: nova, cinder, neutron, glance, keystone, heat, octavia, ironic.
- `entities.vm` MUST be the UUID if present in the incident.
- Keep output lightweight and valid JSON.
"""),
    ("human", "Incident Report: {query}"),
])

In [ ]:
def query_duckdb_logs(sql_query: str) -> str:
    """Execute SQL against DuckDB and return a compact digest string."""
    print(f"[DB QUERY] {sql_query}")
    try:
        rows = db_con.execute(sql_query).fetchall()
        result = format_log_digest(rows)
        print(f"[DB RESULT] {truncate_text(result, 500)}")
        return result
    except Exception as exc:
        print(f"[DB ERROR] {exc}")
        return str(exc)

def query_duckdb_logs_params(sql_query: str, params: list) -> str:
    """Execute a parameterized SQL query and return a compact digest string."""
    print(f"[DB QUERY] {sql_query}")
    print(f"[DB PARAMS] {params}")
    try:
        rows = db_con.execute(sql_query, params).fetchall()
        result = format_log_digest(rows)
        print(f"[DB RESULT] {truncate_text(result, 500)}")
        return result
    except Exception as exc:
        print(f"[DB ERROR] {exc}")
        return str(exc)


# Tool Declerations

In [ ]:
@tool
def search_os_logs(
    service: str = "",
    resource_id: str = "",
    search_terms: str = "",
    time_start: str = "",
    time_end: str = "",
    limit: int = 15,
) -> str:
    """
    Search os_logs and return a compact digest (timestamp|service|level|message…).
    Filters are AND-combined. Prefer resource_id alone first; add 1-2 search_terms
    only to narrow. Optional time_start/time_end accept ISO-like timestamps.
    Default limit is 15 to protect the context window.
    """
    clauses, params = [], []
    if service:
        clauses.append("service = ?")
        params.append(service)
    if resource_id:
        clauses.append("message ILIKE ?")
        params.append(f"%{resource_id}%")
    for term in search_terms.split():
        clauses.append("message ILIKE ?")
        params.append(f"%{term}%")
    if time_start:
        clauses.append("timestamp >= ?")
        params.append(time_start)
    if time_end:
        clauses.append("timestamp <= ?")
        params.append(time_end)
    where = " AND ".join(clauses) if clauses else "1=1"
    capped = max(1, min(int(limit), 30))
    sql = f"""
        SELECT timestamp, service, level, message, source_file
        FROM os_logs
        WHERE {where}
        ORDER BY
          CASE level WHEN 'CRITICAL' THEN 0 WHEN 'ERROR' THEN 1
               WHEN 'WARNING' THEN 2 ELSE 3 END,
          timestamp NULLS LAST
        LIMIT {capped}
    """
    return query_duckdb_logs_params(sql, params)


# Manager Agent and Its State

In [ ]:
investigator_agent = create_agent(
    model=llm,
    tools=[search_os_logs],
    system_prompt="""
    You are an OpenStack RCA investigator. Use search_os_logs iteratively.
    Tool results are already digests (short lines). Do not paste them back in full.

    IMPORTANT — cross-service correlation: after the first hits, extract related
    UUIDs (ports, volumes, networks, images, req-*) and search those in the
    owning service. Prefer resource_id filters and tight time windows.

    Start from any prefetched evidence provided in the user message. Stop once
    you can explain or rule out a root cause, or after a small number of focused
    searches. End with a concise RCA listing services and IDs checked.
    """
)


In [ ]:
def expand_query_node(state: InvestigationState) -> dict:
    """Expand the raw query and prefetch a compact evidence digest."""
    print("--- NODE: Expanding Query ---")
    print(f"[QUERY] raw input: {state['raw_query']}")
    structured_llm = llm.with_structured_output(ExpandedQuery)
    enrichment_chain = ({"query": RunnablePassthrough()} | prompt_expand | structured_llm)
    print("[QUERY] sending prompt to LLM...")
    result = enrichment_chain.invoke(state["raw_query"])
    print(f"[QUERY] result: {result}")

    identifiers = []
    if result.entities and result.entities.resource_id:
        identifiers.append(result.entities.resource_id)
    identifiers.extend(extract_identifiers(state["raw_query"]))
    identifiers.extend(extract_identifiers(result.summary or ""))
    identifiers = list(dict.fromkeys(identifiers))

    services = []
    if result.entities and result.entities.service:
        services.append(result.entities.service)
    for task in result.search_queries or []:
        if task.service:
            services.append(task.service)
    services = list(dict.fromkeys(services))

    time_hint = None
    if result.time_window:
        time_hint = result.time_window.start or result.time_window.end
    if not time_hint:
        # fall back to any clock/date text in the raw query
        from osp_sos_analyser.evidence import extract_evidence_hints
        time_hint = extract_evidence_hints(state["raw_query"]).time_text

    digest = prefetch_evidence_digest(
        db_con,
        identifiers=identifiers,
        services=services,
        keywords=list(result.keywords or [])[:5],
        time_hint=time_hint,
        limit=20,
    )
    print(f"[PREFETCH] {digest[:500]}")
    return {"expanded_plan": result, "prefetch_digest": digest}


###RCA Node

In [ ]:
def synthesize_rca(state: InvestigationState) -> dict:
    plan = state["expanded_plan"]
    evidence = state.get("gathered_evidence", [])
    evidence_text = "\n\n".join(
        f"[{e['service']}] {truncate_text(e.get('raw_logs', ''), 400)}" for e in evidence
    )
    prefetch = truncate_text(state.get("prefetch_digest", ""), 1500)
    rca_prompt = f"""
    Incident: {plan.summary}
    Hypotheses considered: {plan.hypotheses}
    Prefetched digest:
    {prefetch or 'None'}
    Evidence gathered:
    {evidence_text if evidence_text else 'No log evidence was found for any investigated service.'}

    Write a concise root cause analysis. If no evidence was found, say so explicitly
    and suggest what to check next (e.g. widen time window, check neutron/glance, verify UUID).
    """
    result = llm.invoke(rca_prompt)
    return {"final_rca": result.content}


### Add all nodes
workflow.add_node("QUERY_EXPAND", expand_query_node)
workflow.add_node("MANAGER", manage_agents)
workflow.add_node("NOVA_INSPECTOR", nova_inspector)
#### workflow.add_node("CINDER_INSPECTOR", cinder_inspector) # For future

1. The workflow starts by expanding the query
workflow.add_edge(START, "QUERY_EXPAND")
2. After expansion, it ALWAYS goes to the Manager
workflow.add_edge("QUERY_EXPAND", "MANAGER")
3. The Manager routes to a specialist (or ends)
workflow.add_conditional_edges(
    "MANAGER",
    route_from_manager,
    {
        "NOVA_INSPECTOR": "NOVA_INSPECTOR",
        "FINISH": END
    }
)

4. CRITICAL: Specialists ALWAYS report back to the Manager when done!
workflow.add_edge("NOVA_INSPECTOR", "MANAGER")

Compile the application
app = workflow.compile()
print("[WORKFLOW] Graph compiled successfully")


In [ ]:
def run_investigator(state: InvestigationState) -> dict:
    plan = state["expanded_plan"]
    prefetch = state.get("prefetch_digest") or "No prefetched evidence."
    user_message = (
        "Investigate this RHOSP incident using search_os_logs.\n\n"
        f"Incident: {plan.summary}\n"
        f"Known entities: {plan.entities.model_dump()}\n"
        f"Keywords: {plan.keywords}\n"
        f"Time window: {plan.time_window}\n\n"
        f"Prefetched evidence digest (already retrieved — build on it):\n{prefetch}\n"
    )
    raw = investigator_agent.invoke({"messages": [("user", user_message)]})
    messages = compact_message_history(raw.get("messages", []), keep_last=8, tool_result_chars=600)
    final = messages[-1]
    content = getattr(final, "content", None) or (final.get("content") if isinstance(final, dict) else str(final))
    return {"final_rca": content}

workflow = StateGraph(InvestigationState)
workflow.add_node("QUERY_EXPAND", expand_query_node)
workflow.add_node("INVESTIGATOR", run_investigator)
workflow.add_edge(START, "QUERY_EXPAND")
workflow.add_edge("QUERY_EXPAND", "INVESTIGATOR")
workflow.add_edge("INVESTIGATOR", END)
app = workflow.compile()
print("[WORKFLOW] Graph compiled successfully")


# Starting The Inquery

In [ ]:
from langchain_core.utils.uuid import uuid7
initial_state = {
    "raw_query": "Could you check why network port creation failed for port ID 55ab45cf-6925-4811-a008-6fe60d491c5b",
    "expanded_plan": None,
    "prefetch_digest": "",
    "gathered_evidence": [],
    "findings": {},
    "next_node": "",
}

print("🚀 Starting OpenStack Investigation Workflow...\n")
config = {"configurable": {"thread_id": str(uuid7())}}
final_state = app.invoke(initial_state, config={"recursion_limit": 15})

print("\n" + "=" * 40)
print("🎯 INVESTIGATION COMPLETE")
print("=" * 40)

plan = final_state.get("expanded_plan")
if plan:
    print(f"\n[Parsed Intent]: {plan.intent}")
    print(f"[Extracted Time]: {plan.time_window}")

print("\n[Prefetch digest preview]:")
print((final_state.get("prefetch_digest") or "")[:500])

print("\n[Aggregated Findings]:")
for service, data in final_state.get("findings", {}).items():
    print(f"\n--- {service.upper()} ---")
    print(data)
print("\n[Root Cause Analysis]:")
print(final_state.get("final_rca", "No RCA generated."))
